# The Skew-T Log-p Diagram


In [16]:
from datetime import datetime

import matplotlib.pylab as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import metpy.calc as mpcalc
from metpy.plots import Hodograph, SkewT
from metpy.units import units, pandas_dataframe_to_unit_arrays
import numpy as np
from siphon.simplewebservice.wyoming import WyomingUpperAir 

In [17]:
#Pull data from a testy server...

date = datetime(2011, 4, 27, 0)
station = 'BMX'
from time import sleep

for i in range(5):
    try:
        df = WyomingUpperAir.request_data(date, station)
        break
    except Exception as e:
        print(f"Attempt {i+1} failed")
        sleep(5)

Attempt 1 failed
Attempt 2 failed
Attempt 3 failed
Attempt 4 failed
Attempt 5 failed


In [18]:
#Select variables for skewT

data_with_units = pandas_dataframe_to_unit_arrays(df)

# Only want to pull in data up to 100 mb, we can 
# use a boolean array to choose on the desired values
ip100 = data_with_units['pressure'] >= 100 * units.hPa

# Subset pressure and read in other data to only 100 mb
p = data_with_units['pressure'][ip100]
T = data_with_units['temperature'][ip100]
Td = data_with_units['dewpoint'][ip100]
u = data_with_units['u_wind'][ip100]
v = data_with_units['v_wind'][ip100]

## Plotting the Skew-T

MetPy has the ability to make a few specialty plots (with matplotlib as a backend). The skew-T plots are relatively easy to do with only a few lines to plot the key variables and lines on the plot. The main Class that we have to draw a sounding is the `SkewT` object. Here we have rotated the axes and have the ability to draw lines with the `skew.plot()` method, draw wind barbs with the `skew.plot_barbs` method and even plot the special lines of dry adiabats, moist adiabats, and mixing ratio.

https://unidata.github.io/MetPy/latest/api/generated/metpy.plots.SkewT.html#metpy.plots.SkewT

In [ ]:
# Change default to be better for skew-T
fig = plt.figure(figsize=(10, 10))


# Plot the data using normal plotting functions, in this case using
# log scaling in Y, as dictated by the typical meteorological plot
# Add temperature
skew.plot()

#Add dew point
skew.plot()

#Add wind barbs
#1 DRL let's thin out the levels - every third level
#skew.plot_barbs() 

# Add the relevant special lines
skew.plot_dry_adiabats()      
skew.plot_moist_adiabats()
skew.plot_mixing_lines()

#Let's work on our axes
skew.ax.set_ylim()
skew.ax.set_xlim()

#3 Add text
#skew.ax.text()
#4 Add a marker
#skew.ax.plot() 

# Set title using the date and station variables directly
#plt.title(f'{date:%H UTC %d %B %Y} at K{station}') 

#Reduce whitespace of plot
plt.tight_layout()

# Show the plot
plt.show()

## Calculate Parcel Profile

Now we want to use the MetPy function `parcel_profile` to calculated the surfaced-baed CAPE (SBCAPE). We'll need to feed the function with all of the pressure values along with a single starting temperature and dewpoint value. The starting temperature and dewpoint values should be the first value available in each array.

https://unidata.github.io/MetPy/latest/api/generated/metpy.calc.parcel_profile.html#metpy.calc.parcel_profile

In [ ]:
parcel_temp = mpcalc.parcel_profile()

In [ ]:
#Print profile

## Add Parcel Profile to Skew-T and Fill CAPE Area

We can start from our improved skew-T that we generated above, then add a new `skew.plot()` to plot the parcel temperature profile for each level in the pressure variable. Additionally we can use the method available in a `SkewT()` object to shade CAPE values `skew.shade_cape()`. A similar process works for CIN using the appropriate shade method.

https://unidata.github.io/MetPy/latest/api/generated/metpy.plots.SkewT.html#metpy.plots.SkewT.shade_cape

## Add mixing ratio line from the surface dew point upward to the LCL

In [ ]:
#DRL Added 11/4/2025 
from metpy.calc import mixing_ratio
from metpy.calc import vapor_pressure

#DRL diagnostics
#Define pressure levels (in hPa)
pfake = np.linspace(1000, 100, 50) * units.hPa

# Define the sounding mixing ratio at the surface
dewp = Td[0]
pp   =  p[0]

# Step 1: Calculate the water vapor partial pressure (ee) from the dew point
# At the dew point, the air is saturated, so the vapor pressure equals the
# saturation vapor pressure at that temperature.
ee  = mpcalc.saturation_vapor_pressure(dewp)
# Calculate the mixing ratio ww using the vapor pressure and total pressure
ww  = mpcalc.mixing_ratio(ee, pp)

#print(dewp, pp, ee, ww)
print(ee, ww)
print(len(pfake))

In [ ]:
# Calculate the temperature along the mixing ratio line (we need this in order to
# annotate the mixing ratio that intersects the sounding, which identifies the LCL)
# To plot on a skew T, we need pressure and temperature (these are our coordinates)

#Create a temperature range to search over and add units
TT = np.linspace(-60, 40, 500) * units.degC

# For each pressure level, find the temperature where the saturation mixing ratio equals the target
highlight_temps = []   #create a list to store the temperatures

for pressure in pfake:
    # Compute saturation mixing ratio at all temperatures for this pressure
    smrs = mpcalc.saturation_mixing_ratio(pressure, TT)
    
    # Find the temperature where the smr is closest to the target (i.e., the actual mixing ratio)
    idx = np.argmin(np.abs(smrs - ww))
    highlight_temps.append(TT[idx])

# Convert to a unit-aware NumPy array
temperature_profile = np.array([t.magnitude for t in highlight_temps]) * units.degC

print(temperature_profile)

### Plot one last skewT

In [ ]:
#DRL Added 11/4/2025
from metpy.calc import mixing_ratio
from metpy.calc import vapor_pressure

#Paste pretty skewT
fig = plt.figure(figsize=(10, 10))
#Default skewT has isobars, isotherms only
skew= SkewT(fig, rotation = 45)  #Go ahead and run just the skewT without data

# Plot the data using normal plotting functions, in this case using
# log scaling in Y, as dictated by the typical meteorological plot
# Add temperature
skew.plot(p,T,'red', linewidth = 2)

#Add dew point
skew.plot(p, Td, 'green', linewidth = 2)

#Add wind barbs
#DRL let's thin out the levels - every third level
skew.plot_barbs(p[::3],u[::3],v[::3]) 

#******
#1 Add parcel ascent!
#skew.plot(p, parcel_temp, color='black')

#3 Add CAPE (shading)
#skew.shade_cape(p,T, parcel_temp)
#******

#Add the relevant special lines
skew.plot_dry_adiabats(t0=np.arange(-60,240,10)*units.degC, color='darkorange', alpha=0.25)      
skew.plot_moist_adiabats(colors='green', alpha = 0.25)
skew.plot_mixing_lines(pressure=np.arange(1000,99,-20)*units.hPa,linestyle='dotted',
                       colors = 'blue', alpha=0.5)

#Let's work on our axes
skew.ax.set_ylim(1000,100)
skew.ax.set_xlim(-30,40)

#skew.ax.text(-9, 850, '850 hPa',va='center')
#Add marker
#skew.ax.plot(-11, 850, marker = '_', markersize=16, markeredgewidth=2) 

#******
#2 Work on the title
plt.title(f'Date: {date:%H UTC %d %B %Y}', loc='left')
#plt.title(f'Station:K{station}', loc='right')  
#******

#DRL # Plot the highlighted mixing ratio line (see calculation below!)
# skew.plot(pfake, temperature_profile, color='blue', linestyle='--', linewidth=2.5, 
#           label=f'LCL mixing ratio ($g\\,kg^{{-1}}$): {ww.m*1000.:.2f}')
# skew.ax.legend(loc='lower left', frameon=False)


#Reduce whitespace of plot
plt.tight_layout()

# Show the plot
plt.show()

### More Parameters

In [ ]:
from metpy.calc import cape_cin
#From above..
#parcel_temp = mpcalc.parcel_profile(p, T[0], Td[0])

# Compute CAPE and CIN


In [ ]:
#Calculate surface-based LI


In [ ]:
# Calculate the mixed-layer (ML) parcel ascent (e.g., lowest 100 hPA)


In [ ]:
# Redo CAPE/CIN using ML ascent


In [ ]:
# Redo LI
